# Introdução a Banco de Dados - Trabalho Prático 1
---

## Inicialização do Banco

### Criação do Cursor e Conexão

In [46]:
import sqlite3;
conn = sqlite3.connect('database.db');
cursor = conn.cursor();

### Definição das Estruturas

- Tabela hospital

In [47]:
cursor.execute(""" 
    CREATE TABLE hospital 
    (
        CNES        TEXT     NOT NULL,
        NATUREZA    TEXT     NOT NULL,
        GESTAO      TEXT     NOT NULL,
        NAT_JUR     TEXT     NOT NULL,
        PRIMARY KEY(CNES)
    );
""");

- Tabela cid10

In [48]:
cursor.execute(""" 
    CREATE TABLE cid10 
    (
        CID             TEXT        NOT NULL,
        CD_DESCRICAO    TEXT        NOT NULL,
        PRIMARY KEY(CID)
    );
""");

- Tabela procedimentos

In [49]:
cursor.execute(""" 
    CREATE TABLE procedimentos 
    (
        PROC_REA        TEXT        NOT NULL,
        NOME_PROC       TEXT        NOT NULL,
        PRIMARY KEY(PROC_REA)
    );
""");

- Tabela municipios

In [50]:
cursor.execute(""" 
    CREATE TABLE municipios 
    (
        codigo_6d       TEXT        NOT NULL,
        codigo_ibge     TEXT        NOT NULL,
        nome            TEXT        NOT NULL,
        longitude       REAL        NOT NULL,
        latitude        REAL        NOT NULL,
        estado          TEXT        NOT NULL,
        UNIQUE (codigo_ibge),
        PRIMARY KEY(codigo_6d)
    );
""");

- Tabela dado_ibge

In [51]:
cursor.execute(""" 
    CREATE TABLE dado_ibge
    (
        codigo_municipio_completo       TEXT        NOT NULL,
        nome_municipio                  TEXT        NOT NULL,
        sigla_estado                    TEXT        NOT NULL,
        populacao                       REAL        NOT NULL,
        densidade_demografica           REAL        NOT NULL,
        salario_medio                   REAL        NOT NULL,
        mortalidade_infantil            REAL        NOT NULL,
        PRIMARY KEY(codigo_municipio_completo),
        FOREIGN KEY(codigo_municipio_completo) REFERENCES municipios(codigo_ibge)
        ON DELETE CASCADE
    );
""");

- Tabela internacoes

In [52]:
cursor.execute(""" 
    CREATE TABLE internacoes
    (
        N_AIH           TEXT        NOT NULL,
        CNES            TEXT        NOT NULL,
        DT_INTER        TEXT        NOT NULL,
        DT_SAIDA        REAL        NOT NULL,
        QT_DIARIAS      INTEGER     NOT NULL,
        PROC_REA        TEXT        NOT NULL,
        VAL_TOT         REAL        NOT NULL,
        DIAG_PRINC      TEXT        NOT NULL,
        DIAG_SECUN      TEXT        NOT NULL,
        NASC            TEXT        NOT NULL,
        SEXO            INTEGER     NOT NULL,
        IDADE           REAL        NOT NULL,
        MUNIC_RES       TEXT        NOT NULL,
        PRIMARY KEY(N_AIH),
        FOREIGN KEY(CNES) REFERENCES hospital(CNES)
        ON DELETE SET NULL,
        FOREIGN KEY(DIAG_PRINC) REFERENCES cid10(CID)
        ON DELETE SET NULL,
        FOREIGN KEY(PROC_REA) REFERENCES procedimentos(PROC_REA)
        ON DELETE SET NULL,
        FOREIGN KEY(MUNIC_RES) REFERENCES municipios(codigo_6d)
        ON DELETE SET NULL
    );
""");

- Tabela mortes

In [53]:
cursor.execute(""" 
    CREATE TABLE mortes
    (
        N_AIH           TEXT        NOT NULL,
        CID_MORTE       TEXT        NOT NULL,
        PRIMARY KEY(N_AIH),
        FOREIGN KEY(N_AIH) REFERENCES internacoes(N_AIH)
        ON DELETE CASCADE,
        FOREIGN KEY(CID_MORTE) REFERENCES cid10(CID)
        ON DELETE SET NULL
    );
""");

- Tabela condicoes_especificas

In [54]:
cursor.execute(""" 
    CREATE TABLE condicoes_especificas
    (
        N_AIH           TEXT        NOT NULL,
        IND_VDRL        TEXT        NOT NULL,
        PRIMARY KEY(N_AIH),
        FOREIGN KEY(N_AIH) REFERENCES internacoes(N_AIH)
        ON DELETE CASCADE
    );
""");

Salvar alterações no banco

In [55]:
conn.commit();

---
## Instanciação de Mockups

Instanciação de valores falsos para fins de teste.

In [56]:
hospitais = [
    ('2234416', '00', '3', '1244'),
    ('1111111', '20', '1', '3069'),
    ('3333333', '00', '2', '1244')
]
cursor.executemany("INSERT INTO hospital VALUES (?, ?, ?, ?)", hospitais)

cids = [
    ('A15', 'Tuberculose respiratoria, com confirmacao bacteriologica e histologica'),
    ('J189', 'Pneumonia nao especificada'),
    ('I219', 'Infarto agudo do miocardio nao especificado'),
    ('I64', 'Acidente vascular cerebral, nao especificado como hemorragico ou isquemico'),
    ('C50', 'Neoplasia maligna da mama')
]
cursor.executemany("INSERT INTO cid10 VALUES (?, ?)", cids)

procedimentos = [
    ('101010010', 'Consulta medica em atencao especializada'),
    ('40304010', 'Ecocardiografia transtoracica'),
    ('0403010014', 'Tratamento cirurgico de alta complexidade')
]
cursor.executemany("INSERT INTO procedimentos VALUES (?, ?)", procedimentos)

municipios = [
    ('431490', '4314902', 'Porto Alegre', -30.0346, -51.2177, 'RS'),
    ('310620', '3106200', 'Belo Horizonte', -19.9167, -43.9345, 'MG'),
    ('355030', '3550308', 'Sao Paulo', -23.5505, -46.6333, 'SP'),
    ('430510', '4305108', 'Caxias do Sul', -29.1683, -51.1794, 'RS')
]
cursor.executemany("INSERT INTO municipios VALUES (?, ?, ?, ?, ?, ?)", municipios)

dados_ibge = [
    ('4314902', 'Porto Alegre', 'rs', 1332.0, 2800.0, 3.5, 10.0),
    ('3106200', 'Belo Horizonte', 'mg', 2315.0, 7167.0, 3.2, 11.2),
    ('3550308', 'Sao Paulo', 'sp', 11451.0, 7528.0, 4.1, 9.5),
    ('4305108', 'Caxias do Sul', 'rs', 463.0, 282.0, 2.8, 12.1)
]
cursor.executemany("INSERT INTO dado_ibge VALUES (?, ?, ?, ?, ?, ?, ?)", dados_ibge)

internacoes = [
    ('1000000000001', '2234416', '2022-04-09 00:00:00', '2022-04-12 00:00:00', 3, '101010010', 1500.50, 'A15', 'J189', '1990-12-03 00:00:00', 1, 31.0, '431490'),
    ('1000000000002', '1111111', '2023-01-01 00:00:00', '2023-02-05 00:00:00', 35, '40304010', 5000.00, 'I219', 'J189', '1980-05-10 00:00:00', 3, 42.0, '310620'),
    ('1000000000003', '3333333', '2023-03-10 00:00:00', '2023-03-20 00:00:00', 10, '0403010014', 12000.00, 'C50', '', '1975-08-22 00:00:00', 3, 47.0, '355030'),
    ('1000000000004', '2234416', '2023-04-01 00:00:00', '2023-04-06 00:00:00', 5, '101010010', 800.00, 'I219', '', '1960-02-15 00:00:00', 3, 63.0, '430510'),
    ('1000000000005', '1111111', '2023-05-12 00:00:00', '2023-05-14 00:00:00', 2, '40304010', 3500.00, 'I219', 'J189', '1955-11-30 00:00:00', 3, 67.0, '310620'),
    ('1000000000006', '3333333', '2023-06-01 00:00:00', '2023-07-15 00:00:00', 44, '0403010014', 25000.00, 'I64', '', '1948-07-10 00:00:00', 1, 74.0, '355030'),
    ('1000000000007', '2234416', '2023-08-20 00:00:00', '2023-08-21 00:00:00', 1, '101010010', 200.00, 'J189', '', '1995-04-05 00:00:00', 1, 28.0, '431490')
]
cursor.executemany("INSERT INTO internacoes VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", internacoes)

mortes = [
    ('1000000000003', 'C50'),
    ('1000000000004', 'I219'),
    ('1000000000005', 'I219'),
    ('1000000000006', 'I64')
]
cursor.executemany("INSERT INTO mortes VALUES (?, ?)", mortes)

condicoes = [
    ('1000000000001', '0'),
    ('1000000000007', '1')
]
cursor.executemany("INSERT INTO condicoes_especificas VALUES (?, ?)", condicoes)

conn.commit()

---
## Queries Solicitadas

### Queries de Álgebra Relacional

In [58]:
import pandas as pd;

- A1

In [61]:
query = """
SELECT CD_DESCRICAO
FROM cid10
WHERE CID='A15';
"""

df = pd.read_sql_query(query, conn)
df

,CD_DESCRICAO
0,"Tuberculose respiratoria, com confirmacao bact..."


- A2

In [62]:
query = """
SELECT latitude, longitude
FROM municipios
WHERE nome='Porto Alegre';
"""

df = pd.read_sql_query(query, conn)
df

,latitude,longitude
0,-51.2177,-30.0346


- A3

In [63]:
query = """
SELECT NOME_PROC
FROM procedimentos
WHERE PROC_REA='101010010';
"""

df = pd.read_sql_query(query, conn)
df

,NOME_PROC
0,Consulta medica em atencao especializada


- A4

In [64]:
query = """
SELECT N_AIH, CD_DESCRICAO
FROM (
  SELECT *
  FROM (internacoes JOIN cid10 
    ON DIAG_PRINC=CID
    )
  WHERE CNES='2234416'
  );
"""

df = pd.read_sql_query(query, conn)
df

,N_AIH,CD_DESCRICAO
0,1000000000001,"Tuberculose respiratoria, com confirmacao bact..."
1,1000000000004,Infarto agudo do miocardio nao especificado
2,1000000000007,Pneumonia nao especificada


In [66]:
query = """
SELECT nome, populacao
FROM (
  SELECT *
  FROM (municipios JOIN dado_ibge 
    ON codigo_ibge=codigo_municipio_completo
    )
  WHERE estado='RS'
  );
"""

df = pd.read_sql_query(query, conn)
df

,nome,populacao
0,Porto Alegre,1332.0
1,Caxias do Sul,463.0


- A5

### Queries de Linguagem Natural

### Queries para Ponto Extra

---
## Finalização

### Fechando conexão

In [45]:
conn.close()